# TensorQuantLib — Interactive Tutorial

This notebook walks through the complete TensorQuantLib workflow:

1. **Autograd Basics** — Build computation graphs and compute gradients
2. **Black-Scholes Pricing** — Price options and compute Greeks via autodiff
3. **TT-SVD Compression** — Compress multi-dimensional tensors
4. **TT-Surrogate Pricing** — Build a full surrogate for basket options
5. **Visualization** — Plot pricing and Greek surfaces

In [ ]:
# Install if needed (uncomment)
# !pip install -e ".[all]"

import numpy as np
import matplotlib.pyplot as plt

import tensorquantlib as tql
from tensorquantlib.core.tensor import Tensor

print(f"TensorQuantLib v{tql.__version__}")

---
## 1. Autograd Basics

TensorQuantLib includes a custom reverse-mode autodiff engine. Every `Tensor` operation builds a computation graph that can be differentiated.

In [ ]:
# Create tensors with gradient tracking
x = Tensor(np.array([3.0]), requires_grad=True)
y = Tensor(np.array([4.0]), requires_grad=True)

# Build a computation graph: f(x, y) = x^2 * y + y^3
f = x * x * y + y * y * y

print(f"f(3, 4) = {f.item():.1f}")
print(f"Expected: 3^2 * 4 + 4^3 = {3**2 * 4 + 4**3}")

# Backpropagate
f.backward()

print(f"\ndf/dx = {x.grad[0]:.1f}  (expected: 2*x*y = {2*3*4})")
print(f"df/dy = {y.grad[0]:.1f}  (expected: x^2 + 3*y^2 = {3**2 + 3*4**2})")

---
## 2. Black-Scholes Pricing with Autodiff Greeks

Price a European call option and compute Delta (dPrice/dS) automatically.

In [ ]:
from tensorquantlib import bs_price_tensor, bs_price_numpy, bs_delta

# Parameters
S0, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.2

# --- Analytic price ---
price_np = bs_price_numpy(S0, K, T, r, sigma)
delta_analytic = bs_delta(S0, K, T, r, sigma)
print(f"Analytic price:  {price_np:.6f}")
print(f"Analytic Delta:  {delta_analytic:.6f}")

# --- Autodiff price ---
S_t = Tensor(np.array([S0]), requires_grad=True)
price_t = bs_price_tensor(S_t, K=K, T=T, r=r, sigma=sigma)
price_t.backward()

print(f"\nAutodiff price:  {price_t.item():.6f}")
print(f"Autodiff Delta:  {S_t.grad[0]:.6f}")
print(f"\nDelta match: {np.isclose(delta_analytic, S_t.grad[0], atol=1e-6)}")

In [ ]:
# Plot Delta as a function of spot price
spots = np.linspace(60, 140, 200)
deltas = [bs_delta(s, K, T, r, sigma) for s in spots]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(spots, deltas, 'b-', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(K, color='red', linestyle='--', alpha=0.5, label=f'Strike K={K}')
ax.set_xlabel('Spot Price S')
ax.set_ylabel('Delta')
ax.set_title('Black-Scholes Delta vs Spot Price')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. TT-SVD Compression

Compress a multi-dimensional tensor and examine the compression ratio.

In [ ]:
from tensorquantlib import tt_svd, tt_to_full, tt_ranks, tt_memory, tt_error, tt_compression_ratio

# Create a smooth 4D tensor (simulating a pricing surface)
n = 20
x = np.linspace(0, 1, n)
X1, X2, X3, X4 = np.meshgrid(x, x, x, x, indexing='ij')
A = np.exp(-(X1**2 + X2**2 + X3**2 + X4**2))  # Smooth Gaussian

print(f"Original shape: {A.shape}")
print(f"Original size:  {A.nbytes / 1024:.1f} KB")

# Compress with different tolerances
for eps in [1e-2, 1e-4, 1e-8]:
    cores = tt_svd(A, eps=eps)
    ratio = tt_compression_ratio(cores, A.shape)
    error = tt_error(cores, A)
    ranks = tt_ranks(cores)
    print(f"\neps={eps:.0e}: ranks={ranks}, ratio={ratio:.1f}×, error={error:.2e}")

---
## 4. TT-Surrogate for Basket Options

Build a compressed surrogate for a 3-asset basket call option.

In [ ]:
from tensorquantlib import TTSurrogate

# Build a 3-asset basket option surrogate
surr = TTSurrogate.from_basket_analytic(
    S0_ranges=[(80, 120)] * 3,
    K=100, T=1.0, r=0.05,
    sigma=[0.2, 0.25, 0.3],
    weights=[1/3, 1/3, 1/3],
    n_points=30,
    eps=1e-4,
)

# Show diagnostics
surr.print_summary()

In [ ]:
# Evaluate at a specific point
spots = [100.0, 105.0, 95.0]
price = surr.evaluate(spots)
greeks = surr.greeks(spots)

print(f"Spot prices: {spots}")
print(f"Option price: {price:.4f}")
print(f"Deltas: {[f'{d:.4f}' for d in greeks['delta']]}")
print(f"Gammas: {[f'{g:.6f}' for g in greeks['gamma']]}")

In [ ]:
# Evaluate across a range of spot prices for asset 1
spot_range = np.linspace(80, 120, 100)
prices = [surr.evaluate([s, 100.0, 100.0]) for s in spot_range]
deltas_1 = [surr.greeks([s, 100.0, 100.0])['delta'][0] for s in spot_range]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(spot_range, prices, 'b-', linewidth=2)
ax1.set_xlabel('Asset 1 Spot Price')
ax1.set_ylabel('Basket Option Price')
ax1.set_title('Price vs Asset 1 Spot (others at 100)')
ax1.grid(alpha=0.3)

ax2.plot(spot_range, deltas_1, 'r-', linewidth=2)
ax2.set_xlabel('Asset 1 Spot Price')
ax2.set_ylabel('Delta (Asset 1)')
ax2.set_title('Delta vs Asset 1 Spot')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Visualization

Use the built-in visualization functions for 2-asset pricing surfaces.

In [ ]:
from tensorquantlib import plot_pricing_surface, plot_greeks_surface, plot_tt_ranks

# Build a 2-asset surrogate for visualization
surr_2d = TTSurrogate.from_basket_analytic(
    S0_ranges=[(80, 120), (80, 120)],
    K=100, T=1.0, r=0.05,
    sigma=[0.2, 0.25],
    weights=[0.5, 0.5],
    n_points=40,
    eps=1e-4,
)

surr_2d.print_summary()

In [ ]:
# Plot the pricing surface
plot_pricing_surface(surr_2d)

In [ ]:
# Plot Greek surfaces
plot_greeks_surface(surr_2d)

In [ ]:
# Plot TT-rank structure
plot_tt_ranks(surr_2d)

---
## Summary

| Feature | What you learned |
|---------|------------------|
| **Autograd** | Build computation graphs, compute exact gradients |
| **Black-Scholes** | Price options, compute Greeks via autodiff |
| **TT-SVD** | Compress tensors with controllable accuracy |
| **TTSurrogate** | End-to-end basket option surrogate |
| **Visualization** | Publication-quality pricing/Greek surfaces |

For more details, see the [full documentation](https://tensorquantlib.readthedocs.io) and the `examples/` directory.